# LLM Server on Kaggle: Qwen2.5-1.5B-Instruct + vLLM

## 1. Introduction

**vLLM** is an open-source inference and serving engine for large language models. It loads a model onto a GPU and exposes an **OpenAI-compatible HTTP API**, so our local FastAPI backend can call the model in the same way it would call a hosted API.

**Why use a GPU for inference?** A language model performs a huge number of matrix multiplications. A GPU runs thousands of these operations in parallel, so it can generate tokens much faster than a CPU. The model's weights and the key/value (KV) cache also need to live in GPU memory (VRAM).

**What Kaggle provides.** Kaggle Notebooks offer a free NVIDIA GPU (usually two Tesla T4 GPUs, 16 GB each) for a limited session. Kaggle also provides a Linux Python environment with PyTorch and CUDA already installed. We do not need to buy a GPU for this project.

**What this notebook will do.**

1. Confirm that Kaggle gave us a usable CUDA GPU.
2. Check the GPU architecture, Python, PyTorch, and CUDA versions.
3. Install a pinned, compatible version of vLLM.
4. Verify that vLLM imports successfully.

**Loading vs. serving.**

- **Loading** means reading the model's weights from disk into GPU memory and creating a model object. It happens once.
- **Serving** means keeping that loaded model in a long-running process and exposing an API that answers requests. vLLM handles batching, memory management, and token generation for us.

This task stops after installation and verification. The model is **not** downloaded and the server is **not** started yet.

## 2. GPU Verification

Before installing anything, we need to confirm what hardware we actually have.

### CUDA, VRAM, GPU drivers, and compute capability

- **CUDA** is NVIDIA's platform for running general-purpose code on its GPUs. PyTorch and vLLM use CUDA to launch GPU kernels. `torch.version.cuda` shows the CUDA runtime that PyTorch was built with, not the version of the driver.
- **VRAM** (video RAM) is the GPU's own memory. Model weights, activations, and the KV cache must fit inside it. Each T4 has about 16 GB. Running out of VRAM causes an "out of memory" error.
- **A GPU driver** is the low-level software that lets the operating system and CUDA talk to the physical GPU. `nvidia-smi` shows the driver version and the newest CUDA version that driver supports. A new CUDA runtime cannot fix an old driver.
- **Compute capability** identifies the GPU architecture (for example, Turing T4 = 7.5, Pascal P100 = 6.0). vLLM ships compiled CUDA kernels for supported architectures, so the compute capability matters just as much as the amount of VRAM.

The first code cell below stops immediately if CUDA is unavailable. Failing early is better than letting vLLM fail later with a confusing installation error.

In [ ]:
# Fail fast if PyTorch cannot see a CUDA GPU.
# This cell is safe on Kaggle and is not meant to be run on the local Windows machine.
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. In Kaggle, open Settings > Accelerator, "
        "select 'GPU T4 x2', and restart the session before running this notebook."
    )

gpu_index = 0
gpu_name = torch.cuda.get_device_name(gpu_index)
gpu_props = torch.cuda.get_device_properties(gpu_index)

# Report the facts we need for the vLLM compatibility checks.
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
print(f"GPU name: {gpu_name}")
print(f"Total VRAM: {gpu_props.total_memory / 1024**3:.2f} GiB")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version (PyTorch build): {torch.version.cuda}")
print(f"Compute capability: {gpu_props.major}.{gpu_props.minor}")

In [ ]:
# nvidia-smi is installed with the NVIDIA driver, not with pip.
# It shows driver information, GPU utilization, VRAM usage, and running processes.
# The "CUDA Version" in its header is the newest CUDA version the driver supports.
!nvidia-smi

## 3. Environment Compatibility

vLLM is not a pure-Python library. Its wheel contains precompiled C++/CUDA kernels, and those kernels are tied to:

- the **Python ABI** (for example, CPython 3.13),
- an exact **PyTorch** release,
- a **CUDA** runtime build,
- the GPU's **compute capability**.

If any one of these differs, the install may appear to work, but importing vLLM or loading a model can fail with errors such as `undefined symbol`, `libcudart.so`, or `no kernel image is available`.

### Why we pin vLLM 0.20.2

| Requirement | vLLM 0.20.2 | Current Kaggle GPU image |
| --- | --- | --- |
| Operating system | Linux | Linux |
| Python | 3.10 - 3.13 | 3.13 |
| PyTorch | 2.11.0 | 2.11.0 + CUDA 12.8 |
| CUDA runtime | 12.x (`cu129` wheel) | 12.8 |
| GPU compute capability | 7.5 or newer | T4 = 7.5 |
| T4 support | Yes | 2 x Tesla T4 |

**Observed in the current Kaggle environment:** Python 3.13, `PyTorch version: 2.11.0+cu128`, `CUDA runtime (PyTorch build): 12.8`. The previous vLLM 0.19.0 pin expected PyTorch 2.10.0, so the compatibility check correctly stopped instead of force-installing. This notebook now targets vLLM 0.20.2.

We choose **vLLM 0.20.2** because:

- vLLM 0.20.0 through 0.26.0 pin `torch==2.11.0`; vLLM 0.27.0 and newer pin `torch==2.13.0`. Version 0.20.2 is the latest patch in the first compatible line and the closest stable successor to 0.19.0.
- It pins `torch==2.11.0`, which matches the current Kaggle PyTorch release, so pip should not replace Kaggle's PyTorch.
- The vLLM docs for 0.20.2 list compute capability 7.5 or newer, with the T4 explicitly supported. A T4 is exactly 7.5.
- vLLM 0.20.x defaults to a CUDA 13.0 wheel on PyPI. We install the `cu129` variant instead, which vLLM documents as compatible with CUDA 12.x (12.0-12.9) and therefore with Kaggle's CUDA 12.8 runtime.

**Pascal P100 warning:** Kaggle also offers a P100 accelerator. The P100 has compute capability 6.0, which is below vLLM 0.20.2's minimum of 7.5. If the next cells show a P100, switch the Kaggle accelerator to **GPU T4 x2** and restart. Do not try to force vLLM onto unsupported hardware.

We explicitly select `TRITON_ATTN` for the T4 in the model-loading and serving cells. vLLM 0.20.2 otherwise prefers FlashInfer, whose JIT build fails in this Kaggle image because the linker cannot find `libcuda`. `TRITON_ATTN` supports compute capability 7.5 and does not require that runtime compilation.

The next cell only reports the environment. The installation cell later turns these requirements into hard checks.


In [ ]:
# Print the environment facts that affect vLLM compatibility.
import platform
import sys

import torch

print(f"Python version: {sys.version.split()[0]}")
print(f"Operating system: {platform.platform()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA runtime (PyTorch build): {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Visible GPU count: {torch.cuda.device_count()}")
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        print(
            f"  GPU {index}: {props.name} | "
            f"{props.total_memory / 1024**3:.2f} GiB | "
            f"compute capability {props.major}.{props.minor}"
        )
else:
    print("Visible GPU: none")

## 4. Install vLLM

The cell below installs exactly **vLLM 0.20.2** using the **cu129** wheel. It runs only on Kaggle.

What the cell does **before** installing:

1. Refuses to run on Windows or macOS (vLLM does not support them natively).
2. Checks Python 3.10-3.13.
3. Checks that PyTorch is exactly 2.11.0 (ignoring the `+cu128` local suffix).
4. Checks that PyTorch can see CUDA and uses CUDA 12.x.
5. Checks that the GPU's compute capability is at least 7.5 (T4 = 7.5).

If a check fails, the cell raises an error and **stops before pip runs**. That is intentional: a clear compatibility error is easier to fix than a broken installation.

The `%pip install` command:

- Uses the same Python environment as this notebook kernel. This is safer in a notebook than calling `!pip`.
- Pins `vllm==0.20.2+cu129` and adds vLLM's `cu129` wheel index. The plain PyPI 0.20.2 wheel is the CUDA 13.0 build; the `cu129` wheel is the CUDA 12.x-compatible build.
- Does **not** use `--no-deps`, because vLLM needs its matching dependencies.
- Does **not** force or upgrade PyTorch itself. vLLM 0.20.2 declares `torch==2.11.0`, and Kaggle already provides that version with the `+cu128` local suffix.

The download is large (roughly several GB including CUDA libraries and `flashinfer`), so it can take 5-15 minutes. Do not interrupt it.


In [ ]:
# =============================================================================
# KAGGLE ONLY: install the pinned vLLM build.
# Run this cell only after the GPU checks above pass.
# Do not run this cell in the local Windows environment.
# =============================================================================
import sys

import torch

VLLM_VERSION = "0.20.2"
VLLM_WHEEL_VERSION = "0.20.2+cu129"
EXPECTED_TORCH_VERSION = "2.11.0"
MIN_COMPUTE_CAPABILITY = (7, 5)

# vLLM publishes Linux wheels only.
if not sys.platform.startswith("linux"):
    raise RuntimeError(
        "vLLM does not support Windows natively. Run this notebook on Kaggle "
        "(Linux) instead of the local Windows machine."
    )

# vLLM 0.20.2 supports Python 3.10 through 3.13.
if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
    raise RuntimeError(
        f"vLLM {VLLM_VERSION} needs Python 3.10-3.13, but this kernel is "
        f"Python {sys.version.split()[0]}. Do not force the install; use a "
        "Kaggle image with a supported Python version."
    )

# vLLM 0.20.2 is built against PyTorch 2.11.0. Stop instead of letting pip
# replace a different Kaggle PyTorch build.
installed_torch = torch.__version__.split("+")[0]
if installed_torch != EXPECTED_TORCH_VERSION:
    raise RuntimeError(
        f"vLLM {VLLM_VERSION} expects PyTorch {EXPECTED_TORCH_VERSION}, but "
        f"this kernel has PyTorch {torch.__version__}. The Kaggle image may "
        "have changed. Do not force-install. Record the environment output "
        "above and choose a vLLM version that pins this PyTorch release."
    )

# vLLM needs a visible CUDA 12.x build of PyTorch.
if not torch.cuda.is_available():
    raise RuntimeError(
        "PyTorch cannot see a CUDA GPU. In Kaggle, enable Settings > "
        "Accelerator > GPU T4 x2 and restart the session."
    )
if torch.version.cuda is None or not torch.version.cuda.startswith("12."):
    raise RuntimeError(
        f"vLLM {VLLM_VERSION} requires a CUDA 12.x PyTorch build, but PyTorch "
        f"reports CUDA {torch.version.cuda}. Do not force the install."
    )

# vLLM 0.20.2 requires compute capability 7.5 or newer. A T4 is 7.5;
# a P100 is 6.0 and is NOT supported.
gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
if (major, minor) < MIN_COMPUTE_CAPABILITY:
    min_major, min_minor = MIN_COMPUTE_CAPABILITY
    raise RuntimeError(
        f"GPU '{gpu_name}' has compute capability {major}.{minor}, but vLLM "
        f"{VLLM_VERSION} requires {min_major}.{min_minor} or newer. The Tesla "
        "P100 (6.0) is not supported. In Kaggle, switch to 'GPU T4 x2' and "
        "restart the session. Do not attempt random dependency fixes."
    )

print("Compatibility checks passed.")
print(f"  Python: {sys.version.split()[0]}")
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA: {torch.version.cuda}")
print(f"  GPU: {gpu_name} (compute capability {major}.{minor})")
print(f"\nInstalling vLLM {VLLM_WHEEL_VERSION}. This can take 5-15 minutes...")

# vLLM 0.20.x's default PyPI wheel is the CUDA 13.0 build. The cu129 wheel is
# compiled for CUDA 12.9 and is documented as compatible with CUDA 12.x, so it
# works with Kaggle's CUDA 12.8 runtime without replacing PyTorch.
#
# %pip installs into the kernel's Python environment. --no-cache-dir avoids
# filling Kaggle's limited disk with a second copy of the downloaded wheels.
# We deliberately do NOT use --no-deps: vLLM's required dependencies must be
# resolved normally so the compiled kernels and Python packages stay consistent.
%pip install "vllm==0.20.2+cu129" --extra-index-url https://wheels.vllm.ai/0.20.2/cu129 --no-cache-dir --progress-bar off


### Restart the kernel after installation

`%pip` writes the new packages to disk, but the current Python process has already imported PyTorch and loaded native libraries. It may continue using old modules, or it may fail to load vLLM's newly installed CUDA extensions.

After the install finishes successfully:

1. Restart **the notebook kernel**. In Kaggle use the notebook's restart option; in Jupyter or VS Code use **Kernel > Restart**. Do not use "Stop Session" or a factory reset, because that can discard the packages you just installed.
2. Do **not** rerun the install cell.
3. Run the verification cell in Section 5.

If your environment only offers a full reset that discards installed packages, rerun the install cell once after it comes back, then run verification. The important point is that a fresh Python process must import the newly installed packages.

## 5. Verify the Installation

The verification cell:

- imports `vllm`,
- checks that the installed base version is exactly 0.20.2 (the `+cu129` local suffix is allowed),
- checks that PyTorch is still 2.11.0,
- checks that PyTorch still uses CUDA 12.x,
- checks that CUDA is still visible,
- checks that the GPU has compute capability 7.5 or newer,
- prints a success message only when all checks pass.

If the import fails, the error message lists the usual causes. vLLM may print non-fatal warnings about unavailable optimizations (for example, FlashAttention on a T4). A warning is not a failure; a raised exception is.


In [ ]:
# Run this cell AFTER restarting the kernel. Do not rerun the install cell.
import torch

EXPECTED_VLLM_VERSION = "0.20.2"
EXPECTED_TORCH_VERSION = "2.11.0"
MIN_COMPUTE_CAPABILITY = (7, 5)

try:
    import vllm
except Exception as exc:
    raise RuntimeError(
        "vLLM could not be imported. Check the following:\n"
        "1. Did the install cell finish without errors?\n"
        "2. Did you restart the kernel after installing?\n"
        "3. Is a GPU still enabled in Kaggle?\n"
        "4. Does the error mention 'undefined symbol', 'libcudart.so', or "
        "'libcuda.so'? That usually means a stale kernel or a PyTorch/CUDA "
        "binary mismatch.\n"
        "Do not force-install other versions. Copy the full error log."
    ) from exc

installed_version = getattr(vllm, "__version__", "unknown")
# A locally built wheel may report a version such as "0.20.2+cu129".
installed_base_version = installed_version.split("+")[0]
if installed_base_version != EXPECTED_VLLM_VERSION:
    raise RuntimeError(
        f"Expected vLLM {EXPECTED_VLLM_VERSION}, but found {installed_version}. "
        "Rerun the pinned install cell and restart the kernel."
    )

if torch.__version__.split("+")[0] != EXPECTED_TORCH_VERSION:
    raise RuntimeError(
        f"PyTorch changed after installing vLLM: {torch.__version__}. Expected "
        f"{EXPECTED_TORCH_VERSION}. Do not force the install; share this output."
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available after installing vLLM. Enable the Kaggle GPU "
        "accelerator, restart the kernel, and run this cell again."
    )

if torch.version.cuda is None or not torch.version.cuda.startswith("12."):
    raise RuntimeError(
        f"PyTorch is using CUDA {torch.version.cuda}, but this notebook expects "
        "a CUDA 12.x build. Do not force the install; share this output."
    )

major, minor = torch.cuda.get_device_capability(0)
if (major, minor) < MIN_COMPUTE_CAPABILITY:
    min_major, min_minor = MIN_COMPUTE_CAPABILITY
    raise RuntimeError(
        f"GPU compute capability {major}.{minor} is below the minimum "
        f"{min_major}.{min_minor} required by vLLM {EXPECTED_VLLM_VERSION}."
    )

gpu_name = torch.cuda.get_device_name(0)
print(f"vLLM version: {installed_version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU: {gpu_name} (compute capability {major}.{minor})")
print(f"CUDA available: {torch.cuda.is_available()}")
print("\nSUCCESS: vLLM is installed and CUDA is accessible.")
print("The environment is ready for model loading in the next task.")


## 6. What Just Happened?

### During installation

- pip downloaded the vLLM 0.20.2 `cu129` wheel (compiled CUDA kernels), its Python dependencies, `flashinfer`, and CUDA-linked libraries.
- The `cu129` wheel is vLLM's CUDA 12.x-compatible build. The default PyPI 0.20.x wheel is built for CUDA 13.0, which is why the install uses the explicit wheel index.
- Because vLLM 0.20.2 pins `torch==2.11.0`, pip reused Kaggle's existing `torch 2.11.0+cu128` instead of replacing it. The verification cell confirms this.
- No model weights were downloaded. Installing vLLM is like installing an engine; the model itself is not included.
- The kernel restart unloaded the old Python modules so the next process could import the new native libraries cleanly.

### Why vLLM requires compatible GPU hardware

- vLLM's wheel contains machine code compiled for specific GPU architectures. A T4 is Turing (compute capability 7.5). vLLM 0.20.2 requires 7.5 or newer, so the T4 is supported, but Pascal (P100, 6.0) is not.
- CUDA kernels also depend on the CUDA runtime and PyTorch ABI they were built with. Mixing a vLLM wheel with an unrelated PyTorch or CUDA build can produce import errors or crashes, even when the Python package appears to install.
- Newer vLLM releases may target newer CUDA/driver stacks. Pinning a version that matches Kaggle's image is safer and reproducible.

### What model inference means

- The model turns the input text into **tokens** (small pieces of words).
- Each token becomes a vector, and the transformer performs many matrix multiplications to predict the most likely next token.
- The generated token is appended to the input, and the process repeats. This is **autoregressive generation**.
- A GPU speeds up those matrix operations. VRAM holds the model weights, temporary activations, and the KV cache. The KV cache remembers earlier tokens so the model does not recompute the whole prompt for every new token.
- **Serving** keeps the model loaded and answers many requests through an API. That is why we install an inference server instead of reloading the model for every request.


### Why Qwen2.5-1.5B-Instruct?

- **Small enough for a T4.** It has about 1.5 billion parameters; in fp16 the weights are roughly 3 GB, leaving plenty of the T4's 16 GB for the KV cache and activations.
- **Instruction-tuned.** It follows user instructions and is suitable for chat-style RAG answers.
- **Good language coverage.** Qwen2.5 supports many languages and up to 32K tokens of context, which helps when we pass retrieved document chunks.
- **Permissive license.** The Qwen2.5-1.5B-Instruct model is released under the Apache 2.0 license, which is convenient for learning and experimentation.
- **Practical for free hardware.** A 1.5B model starts faster and generates tokens faster on a T4 than a 7B or larger model.

### What the next development step will accomplish

The next task will load `Qwen/Qwen2.5-1.5B-Instruct` with vLLM and start its OpenAI-compatible server on the loopback address inside Kaggle. We will then send test requests to confirm that the server can generate text. The server is not exposed publicly because Kaggle does not permit tunneling. The local FastAPI backend, LangChain, and the RAG pipeline still come later.

For now, stop here: the GPU works, vLLM is installed, and the environment is verified. Do not start the server or download the model yet.

## 7. Understanding Model Loading

**Task 3 starts here.** The previous sections installed and verified vLLM. We can now load the actual Qwen model and run inference.

### 1. What are pretrained model weights?

A language model is a very large mathematical function with millions or billions of adjustable numbers called **parameters** or **weights**. During training, the model sees huge amounts of text and updates those numbers to become better at predicting the next token.

The pretrained weights are the saved result of that training. We are not training Qwen ourselves. We only load the existing weights and use them to generate text. This is called **inference**.

### 2. What is in a Hugging Face model repository?

Hugging Face hosts model repositories that usually contain:

- **Model weights** in files such as `model.safetensors`, often split into several shards.
- **`config.json`**, which describes the architecture, layer sizes, context length, and expected data type.
- **Tokenizer files**, such as the vocabulary, merge rules, and special-token definitions.
- **`tokenizer_config.json`**, which includes Qwen's chat template and tokenizer settings.
- **`generation_config.json`** and license/readme files.

vLLM downloads these files once, caches them locally, and reads them when creating the model.

### 3. What does a tokenizer do?

A **tokenizer** converts text into **tokens**, which are small pieces of words, punctuation, or special markers. Each token has an integer ID from the model's vocabulary. For example, the word "language" might become one token, while a less common word might be split into several.

Qwen is instruction-tuned, so its repository also provides a **chat template**. The template turns a list of messages such as `{"role": "user", "content": "..."}` into the exact format Qwen was trained to see. It adds special markers such as `<|im_start|>` and `<|im_end|>` and opens the assistant turn. After generation, the tokenizer (or vLLM's detokenizer) converts the output token IDs back into readable text.

### 4. What happens when an LLM is loaded into GPU memory?

When vLLM loads the model, it:

1. Downloads the weights if they are not already cached.
2. Converts the weights to the requested precision. We use `float16`, which takes about 2 bytes per parameter.
3. Copies the weights from CPU/disk into **GPU VRAM**.
4. Creates the **KV cache**, the memory that stores attention keys and values for the prompt and generated tokens.
5. Profiles available GPU memory and prepares its inference engine.

For a 1.5B-parameter model, the weights alone are roughly 3 GB in `float16`. The KV cache and working memory need additional space. That is why we reserve 75% of one T4's memory and limit the context to 2048 tokens.

### 5. Loading vs. inference

**Loading** happens once per Python process. It moves the model into GPU memory and can take several minutes on the first run because of the download.

**Inference** happens for every request. The model is already loaded, so inference only tokenizes the prompt, runs the GPU computations, generates new tokens, and decodes them back into text. Keeping the same loaded `llm` object for all tests avoids downloading and loading the model repeatedly.

## 8. Model Configuration

We use the instruction-tuned **Qwen2.5-1.5B-Instruct** model.

Why this model:

- **Small enough for free hardware.** It has about 1.5 billion parameters, roughly 3 GB of weights in `float16`. That fits comfortably on a 16 GB T4 and leaves room for the KV cache.
- **Instruction-tuned.** It follows user questions and is suitable for a chat-style assistant and, later, RAG answers.
- **Native support.** Qwen2.5 is supported by the installed Transformers/vLLM stack, so we can keep `trust_remote_code=False`.
- **Useful context.** Qwen2.5 supports long context, but we deliberately restrict this notebook to 2048 tokens to keep GPU memory use predictable.
- **Practical speed.** A 1.5B model generates faster on a T4 than a 7B or larger model.

We are not using quantization, fine-tuning, or advanced optimization in this task. The next cell only stores the model name in one place so every later cell uses the same model.

In [ ]:
# The Hugging Face model ID. Keeping it in one variable avoids typos and
# makes it easy to see which model the rest of the notebook uses.
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Model: {MODEL_NAME}")

## 9. Load the Model with vLLM

The next cell creates one `LLM` object and keeps it for the rest of the notebook.

### The parameters

- `model=MODEL_NAME`: the Hugging Face repository to load.
- `tensor_parallel_size=1`: use one GPU. The second T4 is left idle because this small model does not need to be split across GPUs.
- `gpu_memory_utilization=0.75`: let vLLM reserve up to 75% of the selected GPU's memory for weights, activations, and KV cache. Leaving headroom is safer than trying to use every byte.
- `max_model_len=2048`: the maximum number of tokens in one request, including the prompt and the generated answer. This limits KV-cache memory.
- `dtype="float16"`: the T4 does not have native hardware support for `bfloat16`, and Qwen's config asks for `bfloat16`. `float16` is the safe T4 choice. This is a precision setting, not quantization.
- `trust_remote_code=False`: Qwen2.5 is supported natively, so we do not need to execute custom code from the model repository.
- `attention_backend="TRITON_ATTN"`: the T4 has compute capability 7.5, while FlashAttention 2 requires 8.0 or newer. vLLM 0.20.2 therefore selects FlashInfer by default, but FlashInfer's runtime JIT build fails in this Kaggle image because the linker cannot find `libcuda`. `TRITON_ATTN` supports the T4 and avoids that JIT step.

The first run downloads about 3 GB of weights and can take several minutes. Later runs reuse the Hugging Face cache.

**Why `TRITON_ATTN`?** The T4 cannot use FlashAttention 2, so vLLM's automatic choice is FlashInfer. FlashInfer tries to compile a missing kernel at load time and needs a `libcuda` linker stub that is absent from this Kaggle image. Selecting `TRITON_ATTN` avoids that compile step and is supported on the T4.

`llm.get_tokenizer()` returns the tokenizer that vLLM already loaded. We reuse it instead of loading a second tokenizer.

### If loading runs out of memory

Do not change several settings at once. First read the full error, then:

1. Confirm no other notebook or process is using the GPU.
2. Restart the kernel to release GPU memory.
3. Try a smaller `max_model_len`, such as 1024.
4. If the GPU is shared or busy, try a slightly lower `gpu_memory_utilization`, such as 0.65.
5. Change one value, rerun this cell once, and compare the result.

If the error says there is "no available memory for the cache blocks", lowering `gpu_memory_utilization` may make it worse; in that specific case, increase it slightly instead. Never tune several parameters blindly.

In [ ]:
# Load the model once. Later cells reuse this same object.
from vllm import LLM

print(f"Loading {MODEL_NAME}...")
print("The first run downloads about 3 GB of weights and can take several minutes.")

llm = LLM(
    model=MODEL_NAME,
    tensor_parallel_size=1,
    gpu_memory_utilization=0.75,
    max_model_len=2048,
    trust_remote_code=False,
    dtype="float16",
    attention_backend="TRITON_ATTN",
)

# Reuse the tokenizer that vLLM loaded with the model.
tokenizer = llm.get_tokenizer()

print(f"Model loaded successfully: {MODEL_NAME}")
print("Ready for inference.")

## 10. Configure Text Generation

`SamplingParams` controls how vLLM chooses the next token. The next cell uses:

- **`temperature=0.7`** controls randomness. Lower values make the model more predictable and focused; higher values make it more varied and creative. A value around 0.7 is a moderate default for chat.
- **`top_p=0.9`** is nucleus sampling. At each step, the model considers the smallest set of most likely tokens whose combined probability is at least 90%, then samples from that set. Lower values are more focused; higher values allow more variety.
- **`max_tokens=150`** caps the number of new tokens. Generation can stop earlier when the model emits an end-of-sequence token. It does not include the prompt tokens.

These settings are reused by every inference cell so the tests are consistent. They are not a claim of optimal quality.

In [ ]:
from vllm import SamplingParams

sampling_params = SamplingParams(
    # Moderate randomness: varied answers without becoming chaotic.
    temperature=0.7,
    # Consider the most likely tokens whose probabilities add up to 90%.
    top_p=0.9,
    # Stop after at most 150 generated tokens.
    max_tokens=150,
)

## 11. Run the First Inference

The next cell asks a simple question and formats it with Qwen's chat template. The template adds the special markers Qwen expects and opens the assistant turn.

Internally, one request goes through these stages:

1. **Chat formatting:** the question is placed into Qwen's message format.
2. **Tokenization:** the tokenizer turns the formatted text into token IDs.
3. **Prefill:** the GPU processes the prompt and fills the KV cache.
4. **Decoding:** the model predicts one token, appends it, and repeats until EOS or `max_tokens`.
5. **Detokenization:** the generated token IDs become readable text.

The `build_chat_prompt()` helper is defined here and reused by the later cells. `add_generation_prompt=True` is important: it tells the template to finish with the assistant marker so the model knows it should answer, not continue the user message.

In [ ]:
def build_chat_prompt(question):
    """Convert a user question into Qwen's chat-formatted prompt."""
    messages = [{"role": "user", "content": question}]
    return tokenizer.apply_chat_template(
        messages,
        # Return text; vLLM will tokenize it with the same tokenizer.
        tokenize=False,
        # Add Qwen's assistant-start marker so the model answers.
        add_generation_prompt=True,
    )


question = "What is a large language model? Explain in three sentences."
prompt = build_chat_prompt(question)

# vLLM accepts a list of prompts and returns one result per prompt.
results = llm.generate([prompt], sampling_params=sampling_params)

# results[0] is the first request; outputs[0] is its first completion.
answer = results[0].outputs[0].text.strip()

print("Question:")
print(question)
print("\nAnswer:")
print(answer)

## 12. Test Multiple Prompts

This cell sends three different questions in one batch. vLLM schedules them together and returns the results in the same order as the prompts.

The first inference may be slower while vLLM warms up its kernels and CUDA graphs. Later requests reuse that work. The model is not reloaded between questions.

In [ ]:
questions = [
    "What is Python?",
    "Explain machine learning in simple words.",
    "What is retrieval-augmented generation?",
]

# Build one chat-formatted prompt per question, then send them as one batch.
prompts = [build_chat_prompt(question) for question in questions]
results = llm.generate(prompts, sampling_params=sampling_params)

for question, result in zip(questions, results):
    answer = result.outputs[0].text.strip()
    print("=" * 80)
    print(f"Question: {question}")
    print(f"Answer: {answer}")

## 13. Measure Inference Time

The next cell times one `llm.generate()` call with `time.perf_counter()`. The measurement includes prompt prefill and token decoding, but not model download/loading or chat-template formatting.

This is a basic measurement, not a rigorous benchmark. The first run may still include one-time warmup, so run the cell a second time if you want a warmer number. Tokens per second is calculated as generated tokens divided by elapsed seconds. It is useful for comparing the same setup, but not for comparing different models or prompts directly.

In [ ]:
import time

question = "Why is the sky blue? Answer in two sentences."
prompt = build_chat_prompt(question)

# Time only the generation call, not prompt formatting or model loading.
start = time.perf_counter()
results = llm.generate([prompt], sampling_params=sampling_params)
elapsed = time.perf_counter() - start

completion = results[0].outputs[0]
answer = completion.text.strip()
generated_tokens = len(completion.token_ids)
tokens_per_second = generated_tokens / elapsed if elapsed > 0 else 0.0

print("Question:")
print(question)
print("\nAnswer:")
print(answer)
print(f"\nTotal generation time: {elapsed:.2f} seconds")
print(f"Generated tokens: {generated_tokens}")
print(f"Approximate tokens per second: {tokens_per_second:.2f}")

## 14. Understanding the Results

### 1. How vLLM executes inference

vLLM turns the formatted prompt into token IDs, runs the model over the prompt (prefill), and stores attention keys and values in the KV cache. It then generates one token at a time (decode), sampling from the next-token probabilities using the temperature and top-p settings. Generation stops at the model's end-of-sequence token or at `max_tokens`. Finally, the token IDs are detokenized into text.

For multiple prompts, vLLM batches requests and uses continuous batching and PagedAttention to keep the GPU busy and manage KV-cache memory efficiently. We do not need to write that scheduling logic ourselves.

### 2. What GPU memory is used for

- **Model weights:** about 3 GB for 1.5B parameters in `float16`.
- **KV cache:** stores attention state for the prompt and generated tokens; it grows with sequence length and batch size.
- **Activations and workspace:** temporary tensors used during the forward pass.
- **CUDA context and libraries:** runtime overhead from PyTorch, CUDA, and vLLM.

`gpu_memory_utilization=0.75` gives vLLM a 75% budget on one GPU. `max_model_len=2048` caps how much KV-cache space a request can require.

### 3. Why the first inference may be slower

The first request can trigger one-time work: CUDA graph capture, kernel compilation or selection, cache initialization, and engine warmup. The first `LLM` load is separate and much slower because it downloads and copies the weights. Later requests reuse the prepared engine, so they are usually faster.

### 4. What tokens per second means

Tokens per second is the number of generated tokens divided by the elapsed generation time. Higher is generally faster, but it is not "words per second": one word can be several tokens. The number also depends on prompt length, output length, sampling settings, GPU load, and whether the request is batched. Compare it only under similar conditions.

### 5. Why inference is not the same as deploying an API

Running `llm.generate()` in a notebook is **offline inference**: one Python process calls the model and receives text. Deploying an API means keeping a server running, accepting HTTP requests, handling concurrency and errors, exposing a stable endpoint, and letting another application call it. Those are separate engineering concerns from the model computation itself.

### 6. How the model will be served next

The next task will start vLLM's OpenAI-compatible server with this model on `127.0.0.1:8000` inside Kaggle and test it with HTTP requests. No public URL or tunnel is created because Kaggle does not permit tunneling. The local FastAPI backend, LangChain, Pinecone, and the Streamlit UI still come later.

## 15. Understanding Model Serving

**Task 4 starts here.** In Task 3, we called the model directly inside the notebook. Now we will keep the same model loaded in a separate server process and call it over HTTP.

> Task 3's closing section anticipated a public tunnel. This task deliberately stops at local serving inside Kaggle. Task 5 adds API-key authentication and documents why Kaggle tunneling is not permitted; it does not connect the local application.

### 1. What is an inference server?

An inference server is a long-running program that keeps a model loaded and waits for requests. A client sends input, the server runs the model, and the server sends the output back. The model is loaded once when the server starts, not for every request.

### 2. Inference vs. serving

- **Inference** is the model computation: tokenize the input, run the transformer, generate tokens, and decode them into text.
- **Serving** is the software around that computation: keeping a process alive, accepting HTTP requests, validating JSON, handling concurrency, reporting health, and shutting down cleanly.

Serving uses inference, but serving is not only inference. It also includes the network and the lifecycle of the server.

### 3. What is an HTTP API?

HTTP is the request/response protocol used by the web. An API defines URLs ("endpoints") and the format of requests and responses. For example:

- `GET /v1/models` asks which models are available.
- `POST /v1/chat/completions` sends a conversation and receives a generated reply.

A request contains a method, a URL, headers, and sometimes a JSON body. A response contains a status code (such as 200 for success) and usually a JSON body.

### 4. What does "OpenAI-compatible" mean?

OpenAI-compatible means that the server implements the same HTTP routes and JSON shapes as OpenAI's API. A client written for OpenAI can change its `base_url` to `http://127.0.0.1:8000/v1` and send requests to our local Qwen model. The requests and responses look familiar, but the model and server are ours.

### 5. Why does vLLM provide an API server?

vLLM already contains the inference engine, batching logic, and GPU memory management. Its OpenAI-compatible server wraps that engine in a web server, so we do not have to rebuild the serving layer. The server loads the model once and reuses it for every request.

### 6. Why don't we need FastAPI here?

vLLM's API server is itself built with FastAPI and Uvicorn internally. Writing another FastAPI app just to call the model would duplicate the routes and add another layer.

Later, the local FastAPI backend will have a different job: it will orchestrate RAG, call this vLLM API, and expose endpoints for the Streamlit UI. For this task, vLLM is the model server.


## 16. Prepare for API Serving

### Important: two model instances would not fit

In Task 3, `LLM(...)` loaded Qwen inside this notebook's Python process. The vLLM API server is a **separate operating-system process** and loads its own copy of the model. Running both at the same time would try to reserve GPU memory for two model instances and two KV caches, which can cause an out-of-memory error on a 16 GB T4.

The cleanup cell below:

1. Checks whether `llm` exists and deletes it.
2. Deletes the tokenizer reference.
3. Runs Python garbage collection.
4. Calls `torch.cuda.empty_cache()`.
5. Prints the remaining PyTorch memory figures.

Deleting Python objects and emptying PyTorch's cache does **not** guarantee that every byte of GPU memory is released. vLLM, CUDA, and the driver may keep allocations alive. Run `!nvidia-smi` to inspect GPU processes. If a large allocation remains, restart the Kaggle kernel from the Kaggle menu, then run the setup cells again.

A kernel restart is not performed automatically. If you already restarted the kernel, no `llm` object exists and this cleanup is harmless.


In [ ]:
# Kaggle only: release the in-notebook model before starting a separate server.
import gc

import torch

if "llm" in globals():
    del llm
    print("Deleted the existing LLM object.")
else:
    print("No LLM object found; nothing to delete.")

if "tokenizer" in globals():
    del tokenizer
    print("Deleted the tokenizer reference.")
else:
    print("No tokenizer reference found.")

# Collect Python objects that may still reference the model or its tensors.
gc.collect()

if torch.cuda.is_available():
    # Release unused cached blocks back to the CUDA allocator.
    torch.cuda.empty_cache()
    allocated_gib = torch.cuda.memory_allocated() / 1024**3
    reserved_gib = torch.cuda.memory_reserved() / 1024**3
    print(f"PyTorch memory allocated: {allocated_gib:.2f} GiB")
    print(f"PyTorch memory reserved: {reserved_gib:.2f} GiB")
    print("If nvidia-smi still shows a large allocation, restart the Kaggle kernel.")
else:
    print("CUDA is not available; no GPU cache to clear.")


## 17. Start the vLLM API Server

The next cell starts the server with `subprocess.Popen()`. A **subprocess** is a separate operating-system process started by Python. It has its own memory and PID.

### Why run it in the background?

A foreground `vllm serve` command would block the notebook cell and prevent us from running readiness and test cells. `Popen()` starts the process without waiting, so the notebook continues.

### What the command does

- Uses `sys.executable`, so it uses the same Python environment where vLLM is installed.
- Uses `-m vllm.entrypoints.openai.api_server`, vLLM's installed API-server entry point.
- Binds to `127.0.0.1`, the loopback address. Only this Kaggle machine can connect; the API is not public.
- Uses port `8000`, a conventional local development port.
- Loads Qwen with one GPU, 75% of GPU memory, a 2048-token context limit, and `float16` for T4 compatibility.
- Uses `--attention-backend TRITON_ATTN` to avoid the FlashInfer JIT failure on the T4, for the same reason as the Task 3 load cell.
- Reads `VLLM_API_KEY` from the environment or from a Kaggle Secret with the same name. The key is passed to the child process through its environment, not on the command line.
- Enables bearer-token authentication. Every `/v1/...` request must include `Authorization: Bearer <API_KEY>`.
- Writes both stdout and stderr to `/kaggle/working/vllm_server.log`.

### Configure the API key safely

1. In Kaggle, open **Add-ons > Secrets**.
2. Create a secret labeled `VLLM_API_KEY`.
3. Generate a long random value on your own computer, for example with `python -c "import secrets; print(secrets.token_urlsafe(32))"`, and paste it into the secret value.
4. Attach the secret to this notebook.
5. Do not paste the key into a notebook cell, print it, or commit it.

Keep `VLLM_API_KEY` separate from `PINECONE_API_KEY`. A client sends `Authorization: Bearer <API_KEY>`; vLLM compares the token and returns `401 Unauthorized` when it is missing or wrong.

`/health` is intentionally not protected by vLLM 0.20.2, so it can still be used to wait for readiness. API-key authentication protects application access, but it does not replace HTTPS or network-level access control. This notebook keeps the server on `127.0.0.1` and does not start a tunnel.

`gpu_memory_utilization` tells vLLM how much of the GPU it may reserve. The log records startup progress, warnings, and errors so failures can be diagnosed.

The next cell does **not** wait for readiness. A process can exist while the model is still loading.


In [ ]:
# Kaggle only: start the vLLM OpenAI-compatible server in the background.
import os
import subprocess
import sys
from pathlib import Path

if "MODEL_NAME" not in globals():
    raise RuntimeError(
        "Run the Task 3 model configuration cell first so MODEL_NAME is defined."
    )

if "server_process" in globals() and server_process.poll() is None:
    raise RuntimeError(
        "A vLLM server is already running. Run the stop-server cell first."
    )

VLLM_API_KEY_NAME = "VLLM_API_KEY"

# Prefer an environment variable. If it is absent, read the Kaggle Secret
# that is attached to this notebook. Never paste the key into a cell.
vllm_api_key = os.environ.get(VLLM_API_KEY_NAME)

if not vllm_api_key:
    try:
        from kaggle_secrets import UserSecretsClient

        vllm_api_key = UserSecretsClient().get_secret(VLLM_API_KEY_NAME)
    except Exception as error:
        raise RuntimeError(
            f"{VLLM_API_KEY_NAME} is not configured. Add it as a Kaggle Secret "
            "or set it in the environment before starting the server."
        ) from error

if not vllm_api_key:
    raise RuntimeError(
        f"{VLLM_API_KEY_NAME} is empty. Add a non-empty value as a Kaggle "
        "Secret or environment variable."
    )

vllm_api_key = vllm_api_key.strip()
if not vllm_api_key:
    raise RuntimeError(
        f"{VLLM_API_KEY_NAME} contains only whitespace. Add a real API key."
    )

# Reuse this header in the authenticated Kaggle test cells.
# Never print this dictionary or the key itself.
auth_headers = {"Authorization": f"Bearer {vllm_api_key}"}

SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8000
BASE_URL = f"http://{SERVER_HOST}:{SERVER_PORT}"
server_log_path = Path("/kaggle/working/vllm_server.log")

# The child process receives the key through its environment. It is not part
# of server_command, so it is not printed or placed in the process arguments.
server_env = os.environ.copy()
server_env[VLLM_API_KEY_NAME] = vllm_api_key

# Replace the previous log so this run is easy to inspect.
with server_log_path.open("w", encoding="utf-8") as log_file:
    server_command = [
        sys.executable,
        "-u",
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--model",
        MODEL_NAME,
        "--host",
        SERVER_HOST,
        "--port",
        str(SERVER_PORT),
        "--tensor-parallel-size",
        "1",
        "--gpu-memory-utilization",
        "0.75",
        "--max-model-len",
        "2048",
        "--dtype",
        "float16",
        "--attention-backend",
        "TRITON_ATTN",
    ]

    print("Starting command:")
    print(" ".join(server_command))
    print("API-key authentication: enabled (key value hidden).")

    # Popen starts the process without blocking this notebook cell.
    server_process = subprocess.Popen(
        server_command,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        env=server_env,
    )

server_pid = server_process.pid

print(f"Started vLLM server with PID {server_pid}.")
print(f"Server URL: {BASE_URL}")
print(f"Log file: {server_log_path}")
print("The model is still loading. Run the readiness cell next.")


## 18. Wait for Server Readiness

Starting a process does not mean the model is ready. The server must import vLLM, load the weights, allocate the KV cache, and start Uvicorn.

The next cell polls `GET /health` every two seconds for up to 180 seconds. It also checks `server_process.poll()` on every loop:

- If the process exited, it prints the last lines of the server log and raises an error.
- If `/health` returns HTTP 200, the API is ready. `/health` is public in vLLM 0.20.2; the `/v1` routes require the bearer token.
- If 180 seconds pass, it prints the log tail and raises a timeout error.

On a first run, downloading the model can take longer than 180 seconds. If the log shows normal download or loading progress, wait and rerun the readiness cell; the server process is still running.


In [ ]:
import time

import requests

if "server_process" not in globals():
    raise RuntimeError("Run the server-start cell first.")


def print_server_log_tail(path, max_lines=40):
    try:
        lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    except FileNotFoundError:
        print("Server log file was not found.")
        return

    print(f"Last {min(max_lines, len(lines))} log lines:")
    print("\n".join(lines[-max_lines:]))


HEALTH_URL = f"{BASE_URL}/health"
timeout_seconds = 180
start_time = time.time()
server_ready = False

while time.time() - start_time < timeout_seconds:
    exit_code = server_process.poll()
    if exit_code is not None:
        print(f"Server exited unexpectedly with code {exit_code}.")
        print_server_log_tail(server_log_path)
        raise RuntimeError("vLLM server failed to start. Read the log output above.")

    try:
        response = requests.get(HEALTH_URL, timeout=5)
        if response.status_code == 200:
            server_ready = True
            break
    except requests.RequestException:
        # The server may still be starting; try again after the short sleep.
        pass

    time.sleep(2)

if not server_ready:
    print(f"Server did not become ready within {timeout_seconds} seconds.")
    print_server_log_tail(server_log_path)
    raise RuntimeError("Server startup timed out. Check the log and rerun this cell.")

print(f"Server is ready: {HEALTH_URL}")


## 19. Verify Available Models

`GET /v1/models` is the OpenAI-compatible endpoint that lists the served models. It confirms which model identifier the API expects. We need that identifier in the chat request.

`/health` tells us the server is alive. `/v1/models` tells us what the API has registered, so it is a useful second check before sending a prompt. The request includes `auth_headers`, which the server-start cell created without printing the key.


In [ ]:
import requests

models_url = f"{BASE_URL}/v1/models"
response = requests.get(models_url, headers=auth_headers, timeout=30)

if not response.ok:
    print(f"HTTP {response.status_code}: {response.text}")
response.raise_for_status()

models = response.json()["data"]
print("Available model identifiers:")
for model in models:
    print(f"- {model['id']}")


## 20. Test Chat Completions

`POST /v1/chat/completions` sends a conversation. The request body has:

- `model`: must match an ID from `/v1/models`.
- `messages`: a list of role/content objects. The user role contains the question.
- `temperature`: 0.7, a moderate amount of randomness.
- `max_tokens`: 150, the output limit.

The request includes the same bearer-token header. vLLM checks it before applying Qwen's chat template, tokenizing the conversation, running inference, and returning an OpenAI-style response. The assistant text is in `choices[0].message.content`.

The response also contains an `id`, `object`, `created`, `model`, `finish_reason`, and `usage` token counts. We print the answer and token usage below.


In [ ]:
import requests

chat_url = f"{BASE_URL}/v1/chat/completions"
question = "Explain what an LLM inference server does in simple words."

payload = {
    "model": MODEL_NAME,
    "messages": [
        {"role": "user", "content": question},
    ],
    "temperature": 0.7,
    "max_tokens": 150,
}

response = requests.post(chat_url, headers=auth_headers, json=payload, timeout=120)

if not response.ok:
    print(f"HTTP {response.status_code}: {response.text}")
response.raise_for_status()

data = response.json()
answer = data["choices"][0]["message"]["content"]
usage = data.get("usage", {})

print("Question:")
print(question)
print("\nAssistant response:")
print(answer)
print("\nToken usage:")
print(usage)


## 21. Test Multiple API Requests

The next cell sends three questions **sequentially** over HTTP to the same already-running server. It does not create a new model or server.

This differs from Task 3's direct `llm.generate()` calls:

- The notebook no longer owns the model; the server process does.
- Each question becomes an HTTP request and a JSON response.
- vLLM applies the chat template on the server.
- The server can handle requests from other clients, not just this notebook.

Because the requests are sequential, the second waits for the first to finish. The server itself is capable of batching concurrent requests, but we are not testing concurrency yet. Every request includes the bearer-token header.


In [ ]:
import requests

chat_url = f"{BASE_URL}/v1/chat/completions"
questions = [
    "What is an HTTP API?",
    "Why does an LLM need GPU memory?",
    "What does OpenAI-compatible mean?",
]

for question in questions:
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "user", "content": question},
        ],
        "temperature": 0.7,
        "max_tokens": 150,
    }

    response = requests.post(chat_url, headers=auth_headers, json=payload, timeout=120)

    if not response.ok:
        print(f"HTTP {response.status_code}: {response.text}")
    response.raise_for_status()

    answer = response.json()["choices"][0]["message"]["content"].strip()
    print("=" * 80)
    print(f"Question: {question}")
    print(f"Answer: {answer}")


## 22. Verify API-Key Authentication

The server now protects `/v1/...` with the `VLLM_API_KEY` secret. This cell tests authentication **inside Kaggle**:

- Request without a key: expect `401 Unauthorized`.
- Request with a wrong key: expect `401 Unauthorized`.
- Request with the valid key: expect `200 OK`.
- `GET /health` without a key: expect `200 OK` because vLLM leaves the readiness route public.

Do not print `auth_headers`, the API key, or the Kaggle Secret. The test only prints HTTP status codes.


In [ ]:
import requests

if "auth_headers" not in globals():
    raise RuntimeError("Run the server-start cell first so the API key is loaded.")

models_url = f"{BASE_URL}/v1/models"
health_url = f"{BASE_URL}/health"

# These requests deliberately use no key, a wrong key, and the real key.
# The output shows only status codes; the key is never printed.
missing_key_response = requests.get(models_url, timeout=30)
wrong_key_response = requests.get(
    models_url,
    headers={"Authorization": "Bearer wrong-test-key"},
    timeout=30,
)
valid_key_response = requests.get(models_url, headers=auth_headers, timeout=30)
health_response = requests.get(health_url, timeout=5)

print(f"Missing key: HTTP {missing_key_response.status_code} (expected 401)")
print(f"Wrong key:   HTTP {wrong_key_response.status_code} (expected 401)")
print(f"Valid key:   HTTP {valid_key_response.status_code} (expected 200)")
print(f"Health:      HTTP {health_response.status_code} (expected 200)")

if missing_key_response.status_code != 401:
    raise RuntimeError("The server accepted a request without an API key.")
if wrong_key_response.status_code != 401:
    raise RuntimeError("The server accepted an invalid API key.")
if valid_key_response.status_code != 200:
    raise RuntimeError("The server rejected the valid API key.")
if health_response.status_code != 200:
    raise RuntimeError("The readiness endpoint is not healthy.")


## 23. Stop the Server

The server is a separate process holding the model in GPU memory. Stopping it makes the API unavailable; any request to port 8000 will fail until you start the server again.

The next cell:

1. Checks whether the process handle exists and the process is still running.
2. Sends a normal termination signal.
3. Waits up to 10 seconds.
4. Uses `kill()` only if the graceful shutdown times out.
5. Prints the result.

We do not run this cell automatically. Keep the server alive while testing, then stop it when finished or before starting another model server.


In [ ]:
import subprocess

if "server_process" not in globals():
    print("No server-process handle found; nothing to stop.")
elif server_process.poll() is not None:
    print(f"Server is already stopped (exit code {server_process.poll()}).")
else:
    print(f"Stopping server PID {server_process.pid}...")
    server_process.terminate()

    try:
        # Give vLLM time to shut down its engine and worker processes.
        server_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        print("Graceful shutdown timed out; forcing termination.")
        server_process.kill()
        server_process.wait()

    print(f"Server stopped (exit code {server_process.returncode}).")

print("The API is no longer available after the server stops.")


## 24. Model-Serving Documentation

### Model-serving lifecycle

1. **Prepare:** remove the in-notebook `LLM` to free GPU memory.
2. **Start:** launch the vLLM API server as a background subprocess.
3. **Wait:** poll `/health` until HTTP 200.
4. **Discover:** call `/v1/models` to confirm the model ID.
5. **Use:** send chat requests to `/v1/chat/completions`.
6. **Stop:** terminate the server when finished.

### Model loading during server startup

The server process loads Qwen from the Hugging Face cache into its own GPU memory. It also creates a new KV cache and starts the vLLM engine. This is why the server needs its own model instance and why the notebook's `LLM` object must be deleted first. The server is ready only after loading finishes and `/health` returns 200.

### HTTP request and response flow

```text
Notebook or client
       |
       | POST /v1/chat/completions (JSON)
       v
vLLM server (FastAPI/Uvicorn)
       |
       | chat template + tokenization
       v
vLLM engine on GPU
       |
       | generated token IDs
       v
detokenization
       |
       | JSON response
       v
Notebook or client
```

### API endpoints used

| Endpoint | Method | Purpose |
| --- | --- | --- |
| `/health` | GET | Check whether the server is ready. |
| `/v1/models` | GET | List the model IDs the API accepts. |
| `/v1/chat/completions` | POST | Send a chat conversation and receive an assistant reply. |

### Inspecting server logs

The log file is `/kaggle/working/vllm_server.log`. In a Kaggle cell, run:

```python
!tail -n 50 /kaggle/working/vllm_server.log
```

or read it with Python:

```python
from pathlib import Path
print(Path("/kaggle/working/vllm_server.log").read_text()[-4000:])
```

The log shows model loading, CUDA setup, warnings, HTTP requests, and errors. If startup fails, read the last 40-50 lines first.

### Why the API only works inside Kaggle

The server binds to `127.0.0.1`, the loopback address. That address is reachable only from the same Kaggle machine. Your Windows computer cannot connect to it directly. This is intentional for Task 4: the model is served locally inside Kaggle, with no public exposure.

The server is loopback-only. Task 5 adds API-key authentication to the vLLM server, but no tunnel is configured because Kaggle does not permit tunneling. The local remote client is prepared but untested against a remote endpoint.

### What Task 5 adds

Task 5 adds bearer-token authentication, a reusable local HTTPS client in `tests/test_remote_llm.py`, and `.env.example` placeholders. It does not expose the Kaggle server: Kaggle does not permit tunneling, so no public URL is created. The FastAPI backend and RAG pipeline are still not implemented.


## 25. Task 5: Remote Access, Authentication, and Kaggle Policy

### Why `127.0.0.1:8000` is not reachable from Windows

`127.0.0.1` is the loopback address. Inside the Kaggle container, it means "this same machine." Your Windows PC has its own loopback address. There is no route from your PC to Kaggle's loopback interface, and Kaggle does not provide a public IP or inbound port for the notebook.

### What an HTTPS tunnel does

An HTTPS tunnel is a client on the Kaggle machine that opens an **outbound** connection to a tunnel provider. Requests from a remote client reach the provider over HTTPS, travel through that outbound connection, and are forwarded to `127.0.0.1:8000`. The provider supplies the public HTTPS certificate. HTTPS encrypts traffic in transit, but it does not prove that the endpoint is safe or that the caller is authorized.

### What a reverse tunnel is

A reverse tunnel also starts from inside the restricted network. Instead of the public internet connecting inward, the Kaggle machine connects outward to a relay. The relay sends traffic back through the already-open connection. This can bypass inbound firewall rules, which is exactly why platforms may prohibit it.

### Why authentication is required

A tunnel URL can be discovered or guessed. Without authentication, anyone who finds it could use your GPU quota or send harmful prompts. The vLLM API key makes the client prove it knows the shared secret:

```http
Authorization: Bearer <API_KEY>
```

The server compares the bearer token and returns `401 Unauthorized` for a missing or wrong key. The key must be long, random, kept in a Kaggle Secret or local `.env`, and never committed. An API key does not replace HTTPS, rate limits, or network-level access control.

### Why Kaggle is not permanent API hosting

Kaggle Notebooks are interactive, quota-limited, and temporary. Sessions stop, public URLs would change, and GPU time is shared. Kaggle's Terms describe personal, non-commercial use, and Kaggle staff have stated that many forms of tunneling are automatically banned to prevent misuse. See:

- https://www.kaggle.com/terms
- https://www.kaggle.com/aup
- https://www.kaggle.com/product-feedback/377918 (Kaggle staff comment on tunneling)
- https://www.kaggle.com/product-feedback/743898 (community report about Kaggle notebooks being used as public LLM servers)

### Decision for this project

No tunnel is configured in this notebook. Kaggle does not permit tunneling. Cloudflare Quick Tunnel, ngrok, SSH tunnels, and similar tools are therefore not used; Cloudflare Quick Tunnel is also not authenticated by default. The vLLM server remains bound to `127.0.0.1`. If remote access is needed later, use a host that explicitly permits it or obtain written permission from Kaggle first. Do not bypass platform restrictions.


## 26. Local Remote Client (Prepared, Not Yet Tested)

`tests/test_remote_llm.py` is a reusable client for a **future permitted HTTPS endpoint**. It is not run against Kaggle, and it has not been tested against any remote endpoint.

It reads `VLLM_API_BASE_URL` and `VLLM_API_KEY` from the local `.env`, rejects non-HTTPS URLs, calls `GET /v1/models`, then `POST /v1/chat/completions`, and prints the generated answer. It handles timeouts, connection failures, and `401` errors without printing the API key.

Local `.env` placeholders:

```env
VLLM_API_BASE_URL=https://your-secure-endpoint.example/v1
VLLM_API_KEY=your_vllm_api_key
```

The script accepts the base URL with or without `/v1` and never creates `/v1/v1`. Run it from the project root with:

```powershell
python tests/test_remote_llm.py
```

Do not point this client at Kaggle through a tunnel. When a permitted endpoint exists, replace the placeholders with its HTTPS URL and key. Expected output from a working endpoint:

```text
Authenticated successfully. Using model: Qwen/Qwen2.5-1.5B-Instruct

Generated response:
A remote LLM API lets an application send prompts to a model running on another machine.
```


## 27. Cleanup

No tunnel process was started, so there is no tunnel PID to stop. Keep using the existing vLLM stop cell (Section 23) to terminate the server when finished. Because tunneling is not configured, only the local Kaggle tests apply.
